# Proyecto Integrador M2

## Nombre del proyecto
**Agente Conversacional de Cobranzas con LangChain 1.x, memoria PostgreSQL y trazabilidad LangSmith**


In [ ]:
!pip install -q langsmith langchain langchain-openai langgraph langgraph-checkpoint-postgres psycopg[binary,pool]==3.2.6 pydantic pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.0/49.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 3.8 MB/s eta 0:00:00


In [ ]:
import os
from pathlib import Path


def leer_secreto(ruta: str, env_var: str | None = None, obligatorio: bool = True) -> str:
    """Lee un secreto desde archivo de Colab o, si no existe, desde variable de entorno."""
    path = Path(ruta)
    if path.exists():
        valor = path.read_text(encoding="utf-8").strip()
    elif env_var and os.getenv(env_var):
        valor = os.getenv(env_var, "").strip()
    else:
        valor = ""

    if obligatorio and not valor:
        raise FileNotFoundError(
            f"No se encontró {ruta}. También puedes definir la variable de entorno {env_var}."
        )
    return valor


# OpenAI
os.environ["OPENAI_API_KEY"] = leer_secreto("/content/api_key.txt", "OPENAI_API_KEY")

# PostgreSQL / Cloud SQL para memoria conversacional
DB_URI = leer_secreto("/content/postgrest_bd.txt", "DB_URI")

# LangSmith
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = leer_secreto("/content/langgraphapi.txt", "LANGSMITH_API_KEY")
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Agente_Cobranzas"
os.environ["LANGCHAIN_PROJECT"] = os.environ["LANGSMITH_PROJECT"]

print("OpenAI API cargada:", bool(os.getenv("OPENAI_API_KEY")))
print("DB_URI cargado para memoria PostgreSQL:", bool(DB_URI))
print("LangSmith activado. Proyecto:", os.environ["LANGSMITH_PROJECT"])

OpenAI API cargada: True
DB_URI cargado para memoria PostgreSQL: True
LangSmith activado. Proyecto: Agente_Cobranzas_Estilo_Docente


In [ ]:
from __future__ import annotations
import re
import json
import uuid
import pandas as pd
from datetime import datetime
from pathlib import Path
from typing import Any, Optional, Literal
from pydantic import BaseModel, Field

from langsmith import Client, traceable
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.tools import tool

from psycopg_pool import ConnectionPool
from langgraph.checkpoint.postgres import PostgresSaver

client = Client()
print("Cliente LangSmith creado correctamente.")

Cliente LangSmith creado correctamente.


In [ ]:
DATA_DIR = Path("/content/data_cobranza")
DATA_DIR.mkdir(parents=True, exist_ok=True)

CLIENTES_PATH = DATA_DIR / "clientes_cobranza.csv"
POLITICAS_PATH = DATA_DIR / "politicas_cobranza.csv"


def crear_dataframes_iniciales() -> tuple[pd.DataFrame, pd.DataFrame]:
    clientes = pd.DataFrame([
        {"dni":"12345678","nombre":"Juan Pérez","producto":"Tarjeta de crédito","deuda_total":5000.00,"dias_atraso":45,"estado":"Mora media","capacidad_pago":"media","segmento":"regular","ultimo_pago":"No registra pago en los últimos 30 días","telefono_valido":True,"canal_preferido":"whatsapp","descuento_max_pct":25,"cuotas_max":6,"opcion_1_descuento_pct":5,"opcion_1_cuotas":1,"opcion_2_descuento_pct":12,"opcion_2_cuotas":3,"opcion_3_descuento_pct":20,"opcion_3_cuotas":6,"flag_reclamo_abierto":False,"fecha_ultimo_contacto":"2026-05-10"},
        {"dni":"87654321","nombre":"María López","producto":"Préstamo personal","deuda_total":12000.00,"dias_atraso":120,"estado":"Mora avanzada","capacidad_pago":"baja","segmento":"vulnerable","ultimo_pago":"Pago parcial hace 75 días","telefono_valido":True,"canal_preferido":"whatsapp","descuento_max_pct":35,"cuotas_max":12,"opcion_1_descuento_pct":10,"opcion_1_cuotas":1,"opcion_2_descuento_pct":20,"opcion_2_cuotas":6,"opcion_3_descuento_pct":30,"opcion_3_cuotas":12,"flag_reclamo_abierto":False,"fecha_ultimo_contacto":"2026-05-08"},
        {"dni":"11223344","nombre":"Carlos Ramírez","producto":"Crédito efectivo","deuda_total":2500.00,"dias_atraso":15,"estado":"Mora inicial","capacidad_pago":"alta","segmento":"preferente","ultimo_pago":"Pago parcial hace 10 días","telefono_valido":True,"canal_preferido":"web","descuento_max_pct":10,"cuotas_max":3,"opcion_1_descuento_pct":0,"opcion_1_cuotas":1,"opcion_2_descuento_pct":5,"opcion_2_cuotas":2,"opcion_3_descuento_pct":10,"opcion_3_cuotas":3,"flag_reclamo_abierto":False,"fecha_ultimo_contacto":"2026-05-12"},
        {"dni":"44556677","nombre":"Ana Torres","producto":"Crédito vehicular","deuda_total":8500.00,"dias_atraso":75,"estado":"Mora alta","capacidad_pago":"media","segmento":"regular","ultimo_pago":"Indica pago realizado, pendiente de validación","telefono_valido":True,"canal_preferido":"whatsapp","descuento_max_pct":20,"cuotas_max":8,"opcion_1_descuento_pct":5,"opcion_1_cuotas":1,"opcion_2_descuento_pct":12,"opcion_2_cuotas":4,"opcion_3_descuento_pct":20,"opcion_3_cuotas":8,"flag_reclamo_abierto":True,"fecha_ultimo_contacto":"2026-05-14"},
        {"dni":"99887766","nombre":"Luis Gómez","producto":"Préstamo personal","deuda_total":1800.00,"dias_atraso":5,"estado":"Mora temprana","capacidad_pago":"alta","segmento":"nuevo","ultimo_pago":"Pago completo hace 35 días","telefono_valido":True,"canal_preferido":"app","descuento_max_pct":0,"cuotas_max":2,"opcion_1_descuento_pct":0,"opcion_1_cuotas":1,"opcion_2_descuento_pct":0,"opcion_2_cuotas":2,"opcion_3_descuento_pct":0,"opcion_3_cuotas":2,"flag_reclamo_abierto":False,"fecha_ultimo_contacto":"2026-05-15"},
    ])

    politicas = pd.DataFrame([
        {"codigo_politica":"P001","categoria":"seguridad","regla":"No solicitar datos sensibles","valor":"CVV, contraseña, número completo de tarjeta","descripcion":"El agente no debe pedir ni almacenar datos de autenticación o tarjeta completa."},
        {"codigo_politica":"P002","categoria":"descuento","regla":"Opciones desde CSV","valor":"opcion_n_descuento_pct y opcion_n_cuotas","descripcion":"El agente debe usar las opciones cargadas en la base, no solicitar al cliente un porcentaje de descuento."},
        {"codigo_politica":"P003","categoria":"descuento","regla":"Descuento fuera de opciones","valor":"derivar a asesor","descripcion":"Si el cliente pide una condición distinta a la disponible, se muestran las opciones del CSV y se indica que un asesor debe validar excepciones."},
        {"codigo_politica":"P004","categoria":"trato_cliente","regla":"Comunicación responsable","valor":"tono claro, respetuoso, empático y no amenazante","descripcion":"Toda respuesta debe orientarse a solución y evitar presión indebida."},
        {"codigo_politica":"P005","categoria":"reclamos","regla":"Deuda no reconocida o pago no aplicado","valor":"derivar a asesor","descripcion":"El agente no debe discutir con el cliente; debe explicar que el caso requiere revisión humana."},
        {"codigo_politica":"P006","categoria":"promesa_pago","regla":"Confirmación obligatoria","valor":"canal oficial","descripcion":"En esta demo el agente no registra promesas; solo informa que debe confirmarse por canal oficial."},
    ])
    return clientes, politicas


def crear_o_actualizar_csv_base() -> None:
    clientes_base, politicas_base = crear_dataframes_iniciales()

    if CLIENTES_PATH.exists():
        clientes_actual = pd.read_csv(CLIENTES_PATH, dtype={"dni": str})
        clientes_actual["dni"] = clientes_actual["dni"].astype(str).str.strip()
        base_por_dni = clientes_base.set_index("dni")
        clientes_actual = clientes_actual.set_index("dni")

        for col in clientes_base.columns:
            if col == "dni":
                continue
            if col not in clientes_actual.columns:
                clientes_actual[col] = clientes_actual.index.map(base_por_dni[col])
            else:
                faltantes = clientes_actual[col].isna()
                clientes_actual.loc[faltantes, col] = clientes_actual.loc[faltantes].index.map(base_por_dni[col])

        clientes_actual = clientes_actual.reset_index()
        clientes_actual.to_csv(CLIENTES_PATH, index=False, encoding="utf-8-sig")
    else:
        clientes_base.to_csv(CLIENTES_PATH, index=False, encoding="utf-8-sig")

    politicas_base.to_csv(POLITICAS_PATH, index=False, encoding="utf-8-sig")


crear_o_actualizar_csv_base()
print("Archivos CSV listos en:", DATA_DIR)


Archivos CSV listos en: /content/data_cobranza


In [ ]:
pd.read_csv(CLIENTES_PATH, dtype={"dni": str}).head()

,dni,nombre,producto,deuda_total,dias_atraso,estado,capacidad_pago,segmento,ultimo_pago,telefono_valido,canal_preferido,descuento_max_pct,cuotas_max,flag_reclamo_abierto,fecha_ultimo_contacto
0,12345678,Juan Pérez,Tarjeta de crédito,5000.0,45,Mora media,media,regular,No registra pago en los últimos 30 días,True,whatsapp,25,6,False,2026-05-10
1,87654321,María López,Préstamo personal,12000.0,120,Mora avanzada,baja,vulnerable,Pago parcial hace 75 días,True,whatsapp,35,12,False,2026-05-08
2,11223344,Carlos Ramírez,Crédito efectivo,2500.0,15,Mora inicial,alta,preferente,Pago parcial hace 10 días,True,web,10,3,False,2026-05-12
3,44556677,Ana Torres,Crédito vehicular,8500.0,75,Mora alta,media,regular,"Indica pago realizado, pendiente de validación",True,whatsapp,20,8,True,2026-05-14
4,99887766,Luis Gómez,Préstamo personal,1800.0,5,Mora temprana,alta,nuevo,Pago completo hace 35 días,True,app,0,2,False,2026-05-15


In [ ]:
def now_iso() -> str:
    return datetime.now().isoformat(timespec="seconds")


def leer_clientes() -> pd.DataFrame:
    df = pd.read_csv(CLIENTES_PATH, dtype={"dni": str})
    df["dni"] = df["dni"].astype(str).str.strip()
    return df


def leer_politicas() -> pd.DataFrame:
    return pd.read_csv(POLITICAS_PATH)


def buscar_cliente(dni: str) -> Optional[dict]:
    dni = str(dni).strip()
    df = leer_clientes()
    row = df[df["dni"].astype(str) == dni]
    if row.empty:
        return None
    return row.iloc[0].to_dict()


def extraer_dni_simple(texto: str) -> Optional[str]:
    match = re.search(r"\b\d{8}\b", str(texto))
    return match.group(0) if match else None


def validar_dni_cliente(dni: Optional[str]) -> dict:
    if not dni:
        return {
            "ok": False,
            "motivo": "dni_no_informado",
            "mensaje": "Para revisar tu caso necesito que me indiques tu DNI de 8 dígitos.",
        }

    dni = str(dni).strip()

    if not re.fullmatch(r"\d{8}", dni):
        return {
            "ok": False,
            "motivo": "formato_invalido",
            "mensaje": "El DNI debe tener 8 dígitos numéricos. Verifica el dato e inténtalo nuevamente.",
        }

    cliente = buscar_cliente(dni)
    if cliente is None:
        return {
            "ok": False,
            "motivo": "dni_no_encontrado",
            "mensaje": (
                f"No encontré información asociada al DNI {dni}. "
                "Verifica el número o solicita apoyo de un asesor por un canal oficial."
            ),
        }

    return {"ok": True, "dni": dni, "cliente": cliente}


def obtener_dni_validado(mensaje: str, dni_contexto: Optional[str] = None) -> dict:
    dni_detectado = extraer_dni_simple(mensaje) or dni_contexto
    return validar_dni_cliente(dni_detectado)


def es_saludo_simple(mensaje: str) -> bool:
    texto = str(mensaje).strip().lower()
    saludos = {"hola", "buenos dias", "buenos días", "buenas tardes", "buenas noches", "hey", "buenas"}
    return texto in saludos


def es_dni_unico(mensaje: str) -> bool:
    return bool(re.fullmatch(r"\s*\d{8}\s*", str(mensaje)))


def es_fuera_de_dominio(mensaje: str) -> bool:
    texto = str(mensaje).lower()

    claves_cobranza = [
        "deuda", "debo", "pago", "pagar", "cuota", "cuotas", "descuento",
        "mora", "atraso", "reclamo", "cobranza", "promesa", "dni",
        "cancelar", "fraccionar", "medio de pago", "medios de pago", "opción", "opcion",
        "asesor", "no reconozco", "ya pagué", "ya pague", "simula", "simular"
    ]

    claves_fuera = [
        "clima", "chiste", "receta", "fútbol", "futbol", "película",
        "pelicula", "programa en python", "historia del perú", "historia del peru",
        "matemática", "matematica", "traduce", "hazme una tarea"
    ]

    tiene_cobranza = any(k in texto for k in claves_cobranza)
    tiene_fuera = any(k in texto for k in claves_fuera)

    return tiene_fuera and not tiene_cobranza


def requiere_identificacion(mensaje: str) -> bool:
    texto = str(mensaje).lower()
    claves = [
        "deuda", "debo", "cuánto debo", "cuanto debo", "opciones", "alternativas",
        "descuento", "cuotas", "promesa", "reclamo", "ya pagué", "ya pague",
        "no reconozco", "pago no aplicado", "mora", "atraso", "simula", "simular",
        "fraccionar", "plan"
    ]
    return any(k in texto for k in claves)


def obtener_opciones_pago_csv(dni: str) -> dict:
    validacion = validar_dni_cliente(dni)
    if not validacion["ok"]:
        return {"ok": False, "mensaje": validacion["mensaje"]}

    cliente = validacion["cliente"]
    deuda = float(cliente["deuda_total"])
    opciones = []

    for idx in [1, 2, 3]:
        desc = float(cliente.get(f"opcion_{idx}_descuento_pct", 0) or 0)
        cuotas = int(cliente.get(f"opcion_{idx}_cuotas", 1) or 1)
        monto_final = round(deuda * (1 - desc / 100), 2)
        monto_cuota = round(monto_final / cuotas, 2)

        opciones.append({
            "opcion": idx,
            "descuento_pct": desc,
            "cuotas": cuotas,
            "monto_final": monto_final,
            "monto_cuota": monto_cuota,
        })

    return {"ok": True, "cliente": cliente, "opciones": opciones}


def formatear_opciones_pago(dni: str) -> str:
    sim = obtener_opciones_pago_csv(dni)
    if not sim.get("ok"):
        return sim["mensaje"]

    cliente = sim["cliente"]
    deuda = float(cliente["deuda_total"])
    producto = cliente["producto"]
    estado = cliente["estado"]
    opciones = sim["opciones"]

    texto = (
        f"Para tu {producto}, registras una deuda referencial de S/ {deuda:,.2f} "
        f"y tu estado actual es {estado}. Estas son las alternativas cargadas para tu perfil:\n\n"
    )

    for op in opciones:
        desc_txt = f"con {op['descuento_pct']:.0f}% de descuento" if op["descuento_pct"] > 0 else "sin descuento"
        texto += (
            f"{op['opcion']}) {desc_txt}: monto final S/ {op['monto_final']:,.2f} "
            f"en {op['cuotas']} cuota(s) de S/ {op['monto_cuota']:,.2f}.\n"
        )

    texto += (
        "\nEstas opciones salen de la base CSV y son referenciales. "
        "Si alguna alternativa te interesa, puedo orientarte para que un asesor la confirme por canal oficial."
    )
    return texto


def simular_opcion_pago_csv(dni: str, opcion: Optional[int] = None) -> str:
    sim = obtener_opciones_pago_csv(dni)
    if not sim.get("ok"):
        return sim["mensaje"]

    if opcion is None:
        return formatear_opciones_pago(dni)

    try:
        opcion_int = int(opcion)
    except Exception:
        return "La opción debe ser 1, 2 o 3."

    if opcion_int not in [1, 2, 3]:
        return "La opción indicada no está disponible. Puedes elegir la opción 1, 2 o 3."

    opciones = {op["opcion"]: op for op in sim["opciones"]}
    op = opciones.get(opcion_int)
    if not op:
        return "La opción indicada no está disponible. Puedes elegir la opción 1, 2 o 3."

    desc_txt = f"{op['descuento_pct']:.0f}% de descuento" if op["descuento_pct"] > 0 else "sin descuento"
    return (
        f"La opción {opcion_int} cargada en la base corresponde a {desc_txt}, "
        f"con monto final S/ {op['monto_final']:,.2f} en {op['cuotas']} cuota(s) "
        f"de S/ {op['monto_cuota']:,.2f}. "
        "Esta simulación es referencial y debe ser confirmada por un asesor antes de formalizarse."
    )


def seleccionar_opcion_pago(dni: str, opcion: int) -> str:
    return simular_opcion_pago_csv(dni, opcion)


def fallback_fuera_dominio() -> str:
    return (
        "Puedo ayudarte solo con consultas de cobranza: deuda, medios de pago, alternativas de pago, "
        "promesas de pago, reclamos o derivación con un asesor. "
        "Indícame tu consulta de cobranza o tu DNI de 8 dígitos para continuar."
    )


In [ ]:
@tool
def consultar_deuda_cliente(dni: str) -> str:
    """
    Consulta la deuda vigente de un cliente validado.

    Args:
        dni: DNI del cliente, con 8 dígitos y existente en el CSV.
    """
    validacion = validar_dni_cliente(dni)
    if not validacion["ok"]:
        return validacion["mensaje"]

    cliente = validacion["cliente"]
    return (
        f"{cliente['nombre']}, tienes una deuda referencial de S/ {float(cliente['deuda_total']):,.2f} "
        f"correspondiente a {cliente['producto']}. "
        f"Actualmente registra {int(cliente['dias_atraso'])} días de atraso ({cliente['estado']}). "
        "También puedo mostrarte medios de pago o alternativas cargadas para tu perfil."
    )


@tool
def generar_alternativas_pago(dni: str) -> str:
    """
    Muestra las alternativas de pago disponibles en el CSV para el cliente.

    Args:
        dni: DNI del cliente, con 8 dígitos y existente en el CSV.
    """
    return formatear_opciones_pago(dni)


@tool
def simular_plan_pago(dni: str, opcion: Optional[int] = None) -> str:
    """
    Simula cuotas usando únicamente las opciones cargadas en el CSV.
    No solicita ni recibe porcentaje de descuento digitado por el cliente.

    Args:
        dni: DNI del cliente, con 8 dígitos y existente en el CSV.
        opcion: Número de opción disponible en el CSV. Si no se informa, se muestran todas las opciones.
    """
    return simular_opcion_pago_csv(dni=dni, opcion=opcion)


@tool
def consultar_medios_y_politicas(tema: str = "general") -> str:
    """
    Consulta medios de pago y políticas de cobranza.

    Args:
        tema: Tema o palabra clave, por ejemplo seguridad, descuento, reclamo, promesa o medios de pago.
    """
    medios = (
        "Medios de pago disponibles: app bancaria, banca por internet, agentes autorizados, "
        "ventanilla y enlace de pago seguro. No compartas contraseñas, CVV ni números completos de tarjeta."
    )

    df = leer_politicas()
    tema_limpio = str(tema).lower().strip()

    if tema_limpio and tema_limpio != "general":
        mask = df.apply(lambda r: tema_limpio in " ".join(map(str, r.values)).lower(), axis=1)
        df_filtrado = df[mask]
    else:
        df_filtrado = df

    if df_filtrado.empty:
        politicas = (
            "Política general: trato respetuoso, no pedir datos sensibles, "
            "no aprobar descuentos definitivos y escalar casos sensibles."
        )
    else:
        politicas = "\n".join(
            f"{r.codigo_politica} | {r.regla}: {r.descripcion}"
            for r in df_filtrado.itertuples(index=False)
        )

    return f"{medios}\n\nPolíticas aplicables:\n{politicas}"


@tool
def gestionar_derivacion_o_promesa(
    dni: str,
    tipo_caso: str,
    descripcion: str,
    opcion: Optional[int] = None,
    fecha_promesa: Optional[str] = None,
    monto_promesa: Optional[float] = None,
) -> str:
    """
    Gestiona selección de alternativa, promesa de pago o derivación por reclamo.
    No registra acuerdos, tickets ni promesas; solo orienta.

    Args:
        dni: DNI del cliente.
        tipo_caso: seleccion_opcion, promesa_pago, reclamo, deuda_no_reconocida o pago_no_aplicado.
        descripcion: Descripción indicada por el cliente.
        opcion: Número de opción elegida, si aplica.
        fecha_promesa: Fecha propuesta, si aplica.
        monto_promesa: Monto propuesto, si aplica.
    """
    validacion = validar_dni_cliente(dni)
    if not validacion["ok"]:
        return validacion["mensaje"]

    cliente = validacion["cliente"]
    tipo = str(tipo_caso).lower().strip()

    if tipo == "seleccion_opcion":
        if opcion is None:
            return "Indica si deseas la opción 1, 2 o 3 para poder orientarte."
        return seleccionar_opcion_pago(dni, int(opcion))

    if tipo == "promesa_pago":
        if not fecha_promesa or monto_promesa is None:
            return "Para orientar una promesa de pago necesito fecha propuesta y monto aproximado."
        monto = float(monto_promesa)
        if monto <= 0:
            return "El monto indicado debe ser mayor a cero."
        return (
            f"Para {cliente['nombre']}, puedo orientar una promesa de pago para el {fecha_promesa} "
            f"por S/ {monto:,.2f}. Esta demo no registra promesas automáticamente; "
            "la confirmación debe realizarse por un canal oficial o con un asesor."
        )

    if tipo in ["reclamo", "deuda_no_reconocida", "pago_no_aplicado"]:
        prioridad = "alta"
        return (
            f"El caso de {cliente['nombre']} requiere revisión humana. "
            f"Tipo de caso: {tipo}. Prioridad referencial: {prioridad}. "
            f"Descripción recibida: {descripcion}. "
            "Esta demo no crea tickets ni guarda información; solo orienta la derivación."
        )

    return (
        "El caso requiere revisión de un asesor. "
        "Esta demo no registra acuerdos, promesas ni tickets; solo orienta la derivación."
    )


tools = [
    consultar_deuda_cliente,
    generar_alternativas_pago,
    simular_plan_pago,
    consultar_medios_y_politicas,
    gestionar_derivacion_o_promesa,
]

print("Herramientas cargadas:", len(tools))
for t in tools:
    print("-", t.name)


Herramientas cargadas: 5
- consultar_deuda_cliente
- generar_alternativas_pago
- simular_plan_pago
- consultar_medios_y_politicas
- gestionar_derivacion_o_promesa


In [ ]:
# ------------------------------------------------------------
#  Modelo principal
# ------------------------------------------------------------
MODEL_PROVIDER = "openai"
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0

modelo = init_chat_model(
    MODEL_NAME,
    model_provider=MODEL_PROVIDER,
    temperature=TEMPERATURE,
)

print("Modelo creado:", MODEL_NAME, "| provider:", MODEL_PROVIDER, "| temperature:", TEMPERATURE)

Modelo creado: gpt-4o-mini | provider: openai | temperature: 0


In [ ]:
# ------------------------------------------------------------
# Memoria PostgreSQL con PostgresSaver
# ------------------------------------------------------------
connection_kwargs = {
    "autocommit": True,
    "prepare_threshold": 0,
}

pool = ConnectionPool(
    conninfo=DB_URI,
    max_size=20,
    kwargs=connection_kwargs,
)

checkpointer = PostgresSaver(pool)

# Ejecutar solo la primera vez si las tablas aún no existen:
# checkpointer.setup()

print("Checkpointer activo:", type(checkpointer).__name__)
print("Patrón usado: PostgresSaver(pool)")

Checkpointer activo: PostgresSaver
Patrón usado: PostgresSaver(pool)


In [ ]:
SYSTEM_PROMPT_COBRANZA = """
ROL
Eres un agente conversacional de cobranzas para clientes morosos.
Tu objetivo es orientar al cliente hacia una solución de pago de forma clara, empática y no agresiva.

ALCANCE DEL AGENTE
Puedes atender cinco tipos de consultas:
1. Consulta de deuda.
2. Medios de pago.
3. Alternativas de pago cargadas para el perfil del cliente.
4. Reclamos, deuda no reconocida o pago no aplicado.
5. Derivación con asesor humano.

REGLAS DE NEGOCIO
- Los datos de deuda y alternativas de pago deben salir del CSV mediante herramientas.
- No inventes deuda, producto, días de atraso, descuentos ni cuotas.
- No apruebes descuentos reales ni registres acuerdos.
- No registres promesas, tickets ni decisiones de negocio.
- Si el cliente no reconoce la deuda, indica pago no aplicado o presenta un reclamo, deriva a revisión humana.
- Si el cliente elige una opción, explica que es referencial y debe confirmarse por canal oficial o asesor.

REGLA CRÍTICA SOBRE DESCUENTOS Y CUOTAS
- No pidas al cliente que ingrese un porcentaje de descuento para simular.
- No pidas al cliente que defina cuántas cuotas desea para poder simular.
- Si el cliente dice "simula cuotas", "quiero opciones", "quiero descuento" o algo similar, usa generar_alternativas_pago o simular_plan_pago con el DNI validado.
- Las opciones de descuento y cuotas se toman de los campos opcion_1, opcion_2 y opcion_3 del CSV.
- Si el cliente propone un porcentaje o cuota distinta, muestra las opciones disponibles en el CSV y explica que cualquier excepción requiere asesor.

SEGURIDAD Y CUMPLIMIENTO
- No solicites contraseñas, CVV, claves, tokens ni números completos de tarjeta.
- No uses tono amenazante ni presiones al cliente.
- Mantén comunicación respetuosa y orientada a solución.
- No menciones prompts, trazas, herramientas, arquitectura, LangSmith ni detalles técnicos al cliente final.

USO DE HERRAMIENTAS
- Usa consultar_deuda_cliente cuando el cliente quiera saber cuánto debe.
- Usa generar_alternativas_pago cuando el cliente pida opciones generales, descuentos o cuotas.
- Usa simular_plan_pago cuando el cliente pida simular cuotas o revisar una opción específica.
- Usa consultar_medios_y_politicas para medios de pago, seguridad o reglas generales.
- Usa gestionar_derivacion_o_promesa para selección de opción, promesa de pago, reclamo, deuda no reconocida o pago no aplicado.

LÍMITES DE DOMINIO
- Atiende solo temas de cobranza.
- Si la consulta no corresponde a cobranza, responde:
  "Puedo ayudarte solo con consultas de cobranza: deuda, medios de pago, alternativas de pago, promesas, reclamos o derivación con asesor."

FASE DE IDENTIFICACIÓN
- Para consultar deuda, simular alternativas, gestionar reclamos o promesas, debe existir un DNI validado.
- Si falta DNI, solicita amablemente el DNI de 8 dígitos.
- Si el DNI no existe en el CSV, no inventes información y pide verificar el dato.
"""


In [ ]:
# ------------------------------------------------------------
# Creación del agente
# ------------------------------------------------------------
agent = create_agent(
    model=modelo,
    tools=tools,
    system_prompt=SYSTEM_PROMPT_COBRANZA,
    checkpointer=checkpointer,
)

print("Agente creado correctamente con create_agent.")
print("Herramientas disponibles:", [t.name for t in tools])
print("Proyecto LangSmith:", os.getenv("LANGSMITH_PROJECT"))

Agente creado correctamente con create_agent.
Herramientas disponibles: ['consultar_deuda_cliente', 'generar_alternativas_pago', 'simular_plan_pago', 'consultar_medios_y_politicas', 'gestionar_derivacion_o_promesa']
Proyecto LangSmith: Agente_Cobranzas_Estilo_Docente


In [ ]:
@traceable(name="procesar_mensaje_cliente", project_name=os.getenv("LANGSMITH_PROJECT", "Agente_Cobranzas_Proyecto"))
def procesar_mensaje_cliente(
    mensaje: str,
    thread_id: str = "cliente_demo_001",
    canal: str = "whatsapp",
    dni_contexto: Optional[str] = None,
) -> str:
    mensaje_limpio = str(mensaje).strip()
    if not mensaje_limpio:
        return "No recibí un mensaje válido. Indica tu consulta sobre deuda, pago, promesa o reclamo."

    if es_fuera_de_dominio(mensaje_limpio):
        return fallback_fuera_dominio()

    validacion_dni = obtener_dni_validado(mensaje_limpio, dni_contexto)
    dni_validado = validacion_dni["dni"] if validacion_dni.get("ok") else None

    if es_saludo_simple(mensaje_limpio) and not dni_validado:
        return (
            "Hola, soy el asistente virtual de cobranzas. "
            "Puedo ayudarte con consultas de deuda, medios de pago, alternativas de pago, promesas o reclamos. "
            "Para revisar tu caso, indícame tu DNI de 8 dígitos."
        )

    if es_dni_unico(mensaje_limpio):
        if not dni_validado:
            return validacion_dni["mensaje"]
        cliente = validacion_dni["cliente"]
        return (
            f"Gracias, {cliente['nombre']}. Ya validé tu DNI. "
            "Puedo ayudarte a consultar tu deuda, revisar medios de pago, simular las alternativas disponibles, "
            "orientar un reclamo o derivarte con un asesor."
        )

    if requiere_identificacion(mensaje_limpio) and not dni_validado:
        return validacion_dni["mensaje"]

    mensajes = []

    if dni_validado:
        mensajes.append({
            "role": "system",
            "content": (
                f"DNI validado contra CSV: {dni_validado}. "
                "Usa este DNI si la consulta requiere identificación. "
                "Para descuentos o cuotas, usa solo las opciones cargadas en el CSV."
            ),
        })

    mensajes.append({"role": "user", "content": mensaje_limpio})

    config = {
        "configurable": {"thread_id": thread_id},
        "tags": ["agente-cobranzas", canal],
        "metadata": {
            "thread_id": thread_id,
            "canal": canal,
            "dni_validado": bool(dni_validado),
            "usa_csv": True,
            "descuentos_desde_csv": True,
            "guarda_negocio": False,
            "checkpoint": "PostgresSaver(pool)",
            "herramientas_visibles": len(tools),
        },
    }

    result = agent.invoke({"messages": mensajes}, config=config)
    return result["messages"][-1].content


In [ ]:
# ------------------------------------------------------------
# Verificación de ambiente
# ------------------------------------------------------------
print("LANGSMITH_TRACING:", os.getenv("LANGSMITH_TRACING"))
print("LANGCHAIN_TRACING_V2:", os.getenv("LANGCHAIN_TRACING_V2"))
print("LANGSMITH_PROJECT:", os.getenv("LANGSMITH_PROJECT"))
print("LANGSMITH_API_KEY existe:", bool(os.getenv("LANGSMITH_API_KEY")))
print("OPENAI_API_KEY existe:", bool(os.getenv("OPENAI_API_KEY")))
print("Checkpointer:", type(checkpointer).__name__)
print("CLIENTES_PATH:", CLIENTES_PATH)
print("POLITICAS_PATH:", POLITICAS_PATH)


def verificar_langsmith():
    try:
        cliente_ls = Client()
        list(cliente_ls.list_projects(limit=1))
        print("Cliente LangSmith operativo.")
        print("Proyecto esperado:", os.getenv("LANGSMITH_PROJECT"))
        return cliente_ls
    except Exception as exc:
        print("No se pudo validar LangSmith. Revisa /content/langgraphapi.txt y conexión.")
        print("Detalle:", exc)
        return None


verificar_langsmith()

LANGSMITH_TRACING: true
LANGCHAIN_TRACING_V2: true
LANGSMITH_PROJECT: Agente_Cobranzas_Estilo_Docente
LANGSMITH_API_KEY existe: True
OPENAI_API_KEY existe: True
Checkpointer: PostgresSaver
CLIENTES_PATH: /content/data_cobranza/clientes_cobranza.csv
POLITICAS_PATH: /content/data_cobranza/politicas_cobranza.csv
Cliente LangSmith operativo.
Proyecto esperado: Agente_Cobranzas_Estilo_Docente


Client (API URL: https://api.smith.langchain.com)

In [ ]:
leer_clientes().head()

,dni,nombre,producto,deuda_total,dias_atraso,estado,capacidad_pago,segmento,ultimo_pago,telefono_valido,canal_preferido,descuento_max_pct,cuotas_max,flag_reclamo_abierto,fecha_ultimo_contacto
0,12345678,Juan Pérez,Tarjeta de crédito,5000.0,45,Mora media,media,regular,No registra pago en los últimos 30 días,True,whatsapp,25,6,False,2026-05-10
1,87654321,María López,Préstamo personal,12000.0,120,Mora avanzada,baja,vulnerable,Pago parcial hace 75 días,True,whatsapp,35,12,False,2026-05-08
2,11223344,Carlos Ramírez,Crédito efectivo,2500.0,15,Mora inicial,alta,preferente,Pago parcial hace 10 días,True,web,10,3,False,2026-05-12
3,44556677,Ana Torres,Crédito vehicular,8500.0,75,Mora alta,media,regular,"Indica pago realizado, pendiente de validación",True,whatsapp,20,8,True,2026-05-14
4,99887766,Luis Gómez,Préstamo personal,1800.0,5,Mora temprana,alta,nuevo,Pago completo hace 35 días,True,app,0,2,False,2026-05-15


In [ ]:
leer_politicas().head(10)

,codigo_politica,categoria,regla,valor,descripcion
0,P001,seguridad,No solicitar datos sensibles,"CVV, contraseña, número completo de tarjeta",El agente no debe pedir ni almacenar datos de ...
1,P002,descuento,Descuento automático máximo,Según campo descuento_max_pct por cliente,El agente solo puede simular descuentos dentro...
2,P003,descuento,Descuento mayor al límite,requiere_revision=True,Si el cliente solicita un descuento superior a...
3,P004,trato_cliente,Comunicación responsable,"tono claro, respetuoso, empático y no amenazante",Toda respuesta debe orientarse a solución y ev...
4,P005,reclamos,Deuda no reconocida o pago no aplicado,derivar a asesor,El agente no debe discutir con el cliente; deb...
5,P006,promesa_pago,Confirmación obligatoria,pedir confirmación antes de registrar,En esta demo el agente no registra promesas; s...


# Pruebas funcionales

Estas pruebas sustentan la demo:

- saludo con fase de identificación,
- validación de DNI,
- consulta de deuda,
- simulación de alternativas desde CSV,
- selección de opción,
- solicitud de descuento sin pedir porcentaje al cliente,
- reclamo o pago no aplicado,
- memoria por `thread_id`,
- DNI inválido o inexistente,
- consulta fuera de dominio.


In [ ]:
respuesta_1 = procesar_mensaje_cliente(
    "Hola",
    thread_id="demo_cliente_12345678",
    canal="whatsapp",
)
print(respuesta_1)


Hola, soy el asistente virtual de cobranzas. Puedo ayudarte con consultas de deuda, medios de pago, alternativas de pago, promesas o reclamos. Para revisar tu caso, indícame tu DNI de 8 dígitos.


In [ ]:
respuesta_2 = procesar_mensaje_cliente(
    "12345678",
    thread_id="demo_cliente_12345678",
    canal="whatsapp",
)
print(respuesta_2)


Juan Pérez, tienes una deuda referencial de S/ 5,000.00 correspondiente a tu tarjeta de crédito. Actualmente registras 45 días de atraso (Mora media). Si deseas, puedo ayudarte a revisar medios de pago o simular alternativas de pago. ¿Qué te gustaría hacer?


In [ ]:
respuesta_3 = procesar_mensaje_cliente(
    "Quiero saber cuánto debo.",
    thread_id="demo_cliente_12345678",
    canal="whatsapp",
    dni_contexto="12345678",
)
print(respuesta_3)


Para tu tarjeta de crédito, aquí tienes algunas opciones referenciales debido a tus problemas con el pago:

1) Pagar con 5% de descuento: monto final S/ 4,750.00 en 1 cuota de S/ 4,750.00.
2) Pagar con 12% de descuento: monto final S/ 4,400.00 en 3 cuotas de S/ 1,466.67.
3) Pagar con 20% de descuento: monto final S/ 4,000.00 en 6 cuotas de S/ 666.67.

Estas opciones son simulaciones referenciales. Si alguna te interesa, puedo derivarte con un asesor para confirmar condiciones y formalizar la modificación. ¿Te gustaría elegir alguna opción?


In [ ]:
respuesta_4 = procesar_mensaje_cliente(
    "Simula cuotas para mi caso.",
    thread_id="demo_cliente_12345678",
    canal="whatsapp",
    dni_contexto="12345678",
)
print(respuesta_4)


Te interesa la opción 2: pagar con un 12% de descuento, lo que te dejaría un monto final de S/ 4,400.00 en 3 cuotas de S/ 1,466.67. Esta alternativa es referencial y debe ser confirmada. Te derivaré con un asesor para validar las condiciones.


In [ ]:
respuesta_5 = procesar_mensaje_cliente(
    "Me interesa la opción 2.",
    thread_id="demo_cliente_12345678",
    canal="whatsapp",
    dni_contexto="12345678",
)
print(respuesta_5)


Con un 15% de descuento, el monto final de tu deuda de S/ 5,000.00 sería S/ 4,250.00, pagadero en 3 cuotas de S/ 1,416.67 cada una. Esta alternativa se encuentra dentro de los límites referenciales de tu perfil, pero es importante que se confirme con un asesor. ¿Te gustaría que te derive con uno para validar esta opción?


In [ ]:
respuesta_6 = procesar_mensaje_cliente(
    "¿Puedo acceder a un descuento?",
    thread_id="demo_cliente_12345678",
    canal="whatsapp",
    dni_contexto="12345678",
)
print(respuesta_6)


Tu caso ha sido registrado para revisión humana, ya que mencionas que has pagado pero sigues recibiendo cobros. Un asesor se pondrá en contacto contigo para resolver esta situación. Si necesitas más asistencia, no dudes en decírmelo.


In [ ]:
respuesta_7 = procesar_mensaje_cliente(
    "Mi DNI es 44556677. Yo ya pagué y me siguen cobrando. Quiero presentar un reclamo.",
    thread_id="demo_cliente_44556677",
    canal="whatsapp",
)
print(respuesta_7)


La última vez que revisamos tu caso, discutimos la opción de pagar con un 15% de descuento, lo que te dejaría un monto final de S/ 4,250.00, pagadero en 3 cuotas de S/ 1,416.67 cada una. Esta alternativa está dentro de los límites referenciales de tu perfil, pero debe ser confirmada con un asesor. ¿Te gustaría proceder con esta opción o necesitas más información?


In [ ]:
respuesta_8 = procesar_mensaje_cliente(
    "Quiero saber mi deuda. Mi DNI es 99999999.",
    thread_id="demo_cliente_invalido",
    canal="whatsapp",
)
print(respuesta_8)


No encontré información asociada al DNI 99999999. Verifica el número o solicita apoyo de un asesor por un canal oficial.


In [ ]:
respuesta_9 = procesar_mensaje_cliente(
    "Cuéntame un chiste sobre fútbol.",
    thread_id="demo_fuera_dominio",
    canal="whatsapp",
)
print(respuesta_9)


Puedo ayudarte solo con consultas de cobranza: deuda, medios de pago, alternativas de pago, promesas de pago, reclamos o derivación con un asesor. Indícame tu consulta de cobranza o tu DNI de 8 dígitos para continuar.


In [ ]:
# ------------------------------------------------------------
# Chat interactivo
# ------------------------------------------------------------
def _extraer_contenido_mensaje(msg: Any) -> str:
    """
    Extrae el contenido textual de un mensaje LangChain/LangGraph.
    Soporta objetos BaseMessage y diccionarios serializados.
    """
    if msg is None:
        return ""

    if isinstance(msg, dict):
        contenido = msg.get("content", "")
    else:
        contenido = getattr(msg, "content", "")

    if isinstance(contenido, list):
        partes = []
        for item in contenido:
            if isinstance(item, dict):
                partes.append(str(item.get("text") or item.get("content") or ""))
            else:
                partes.append(str(item))
        return " ".join(p for p in partes if p).strip()

    return str(contenido or "").strip()


def _extraer_rol_mensaje(msg: Any) -> str:
    """
    Normaliza el rol del mensaje recuperado desde el estado del agente.
    """
    if isinstance(msg, dict):
        rol = msg.get("role") or msg.get("type") or ""
    else:
        rol = getattr(msg, "type", "") or getattr(msg, "role", "")

    rol = str(rol).lower().strip()

    if rol in {"human", "user"}:
        return "cliente"
    if rol in {"ai", "assistant"}:
        return "agente"
    if rol == "system":
        return "sistema"
    if rol == "tool":
        return "herramienta"
    return rol or "desconocido"


def reconstruir_historial_desde_checkpointer(thread_id: str) -> list[dict]:
    """
    Reconstruye el historial conversacional desde el estado persistido del agente.

    Punto importante:
    - No usa la lista local del chat.
    - Recupera el estado mediante agent.get_state(config), usando el mismo thread_id.
    - Funciona si el notebook se reinicia y luego se retoma la conversación con el mismo thread_id,
      siempre que el checkpointer de PostgreSQL esté disponible.
    """
    config = {"configurable": {"thread_id": thread_id}}

    try:
        estado = agent.get_state(config)
    except Exception as exc:
        print(f"Advertencia: no se pudo leer el estado persistido del agente: {exc}")
        return []

    valores = getattr(estado, "values", None)
    if valores is None and isinstance(estado, dict):
        valores = estado.get("values", estado)

    if not isinstance(valores, dict):
        return []

    mensajes = valores.get("messages", []) or []
    historial = []

    for msg in mensajes:
        rol = _extraer_rol_mensaje(msg)
        contenido = _extraer_contenido_mensaje(msg)

        # Para el resumen ejecutivo se excluyen mensajes técnicos.
        if rol in {"sistema", "herramienta"}:
            continue
        if not contenido:
            continue

        historial.append({
            "rol": rol,
            "mensaje": contenido,
            "fuente": "postgres_checkpointer",
        })

    return historial


def _inferir_dni_desde_historial(historial: list[dict], dni_contexto: Optional[str] = None) -> Optional[str]:
    """
    Intenta recuperar un DNI válido desde el contexto explícito o desde mensajes persistidos.
    Solo acepta DNIs existentes en el CSV.
    """
    if dni_contexto:
        validacion = validar_dni_cliente(dni_contexto)
        if validacion.get("ok"):
            return str(dni_contexto)

    texto_total = " ".join(h.get("mensaje", "") for h in historial)
    candidatos = re.findall(r"\b\d{8}\b", texto_total)

    for dni in candidatos:
        validacion = validar_dni_cliente(dni)
        if validacion.get("ok"):
            return dni

    return None


def resumir_sesion_desde_historial(historial: list[dict], dni: Optional[str] = None) -> str:
    """
    Genera un resumen determinístico a partir de un historial ya reconstruido.
    No usa LLM y no registra información de negocio.
    """
    mensajes_cliente = [h for h in historial if h.get("rol") == "cliente"]
    total_turnos = len(mensajes_cliente)

    texto_total = " ".join(h.get("mensaje", "").lower() for h in historial)
    temas = []

    if any(k in texto_total for k in ["deuda", "debo", "cuánto debo", "cuanto debo", "saldo", "monto"]):
        temas.append("consulta de deuda")
    if any(k in texto_total for k in ["opciones", "alternativas", "descuento", "cuotas", "fraccionar", "plan"]):
        temas.append("alternativas de pago")
    if any(k in texto_total for k in ["reclamo", "ya pagué", "ya pague", "no reconozco", "pago no aplicado"]):
        temas.append("reclamo o derivación")
    if any(k in texto_total for k in ["promesa", "pagaré", "pagare", "me comprometo"]):
        temas.append("promesa de pago")
    if any(k in texto_total for k in ["medio de pago", "medios de pago", "canal", "banco", "yape", "plin"]):
        temas.append("medios de pago")

    temas_txt = ", ".join(sorted(set(temas))) if temas else "orientación general de cobranza"
    dni_txt = dni if dni else "no validado"

    if total_turnos == 0:
        return (
            "Resumen de sesión: no se encontraron mensajes del cliente en la memoria persistente para este thread_id. "
            f"DNI en contexto: {dni_txt}. No se registraron acuerdos, promesas ni tickets."
        )

    return (
        f"Resumen de sesión: se recuperaron {total_turnos} mensaje(s) del cliente desde la memoria persistente. "
        f"DNI en contexto: {dni_txt}. Tema(s) revisado(s): {temas_txt}. "
        "No se registraron acuerdos, promesas ni tickets; solo se brindó orientación referencial."
    )


def resumir_sesion_persistente(
    thread_id: str,
    dni_contexto: Optional[str] = None,
    historial_respaldo: Optional[list[dict]] = None,
) -> str:
    """
    Resumen de cierre basado primero en la memoria persistente del agente.

    Orden de uso:
    1. Intenta reconstruir el historial desde PostgreSQL/PostgresSaver usando thread_id.
    2. Si no encuentra mensajes persistidos, usa historial_respaldo solo como fallback de demo.
    """
    historial_persistente = reconstruir_historial_desde_checkpointer(thread_id)

    if historial_persistente:
        dni = _inferir_dni_desde_historial(historial_persistente, dni_contexto)
        return resumir_sesion_desde_historial(historial_persistente, dni=dni)

    historial_respaldo = historial_respaldo or []
    dni = _inferir_dni_desde_historial(historial_respaldo, dni_contexto)

    resumen = resumir_sesion_desde_historial(historial_respaldo, dni=dni)
    return resumen + " Nota técnica: se usó historial local como respaldo porque no se pudo recuperar estado persistido."


@traceable(name="chat_interactivo_cobranzas", project_name=os.getenv("LANGSMITH_PROJECT", "Agente_Cobranzas_Proyecto"))
def iniciar_chat_cobranzas(
    thread_id: str = "chat_cliente_12345678",
    canal: str = "whatsapp",
) -> None:
    print("=" * 80)
    print("CHAT INTERACTIVO - AGENTE DE COBRANZAS")
    print("=" * 80)
    print(f"Thread ID: {thread_id}")
    print(f"Canal: {canal}")
    print("Escribe XEND para salir.")
    print("-" * 80)
    print(
        "Agente: Hola, soy el asistente virtual de cobranzas. "
        "Para revisar tu caso, indícame tu DNI de 8 dígitos."
    )
    print("-" * 80)

    contexto_sesion = {"dni": None, "historial": []}

    while True:
        mensaje = input("Cliente: ").strip()

        if mensaje.upper() == "XEND":
            resumen = resumir_sesion_persistente(
                thread_id=thread_id,
                dni_contexto=contexto_sesion.get("dni"),
                historial_respaldo=contexto_sesion["historial"],
            )
            print("\nAgente:", resumen)
            print("Chat finalizado por el usuario.")
            break

        if not mensaje:
            print("Agente: Ingresa un mensaje válido o escribe XEND para salir.")
            continue

        dni_detectado = extraer_dni_simple(mensaje)
        if dni_detectado:
            validacion = validar_dni_cliente(dni_detectado)
            if validacion["ok"]:
                contexto_sesion["dni"] = dni_detectado

        contexto_sesion["historial"].append({"rol": "cliente", "mensaje": mensaje, "timestamp": now_iso()})

        try:
            respuesta = procesar_mensaje_cliente(
                mensaje=mensaje,
                thread_id=thread_id,
                canal=canal,
                dni_contexto=contexto_sesion.get("dni"),
            )

            contexto_sesion["historial"].append({"rol": "agente", "mensaje": respuesta, "timestamp": now_iso()})

            print("\nAgente:", respuesta)
            print("-" * 80)

        except KeyboardInterrupt:
            resumen = resumir_sesion_persistente(
                thread_id=thread_id,
                dni_contexto=contexto_sesion.get("dni"),
                historial_respaldo=contexto_sesion["historial"],
            )
            print("\nAgente:", resumen)
            print("Chat interrumpido manualmente.")
            break

        except Exception as exc:
            print(f"Agente: Ocurrió un error controlado: {exc}")
            print("Puedes continuar escribiendo o digitar XEND para salir.")
            print("-" * 80)


# Para iniciar la conversación, ejecuta:
# iniciar_chat_cobranzas(thread_id="chat_cliente_12345678", canal="whatsapp")



In [ ]:
iniciar_chat_cobranzas(thread_id="chat_cliente_12345678", canal="whatsapp")

CHAT INTERACTIVO - AGENTE DE COBRANZAS
Thread ID: chat_cliente_12345678
Canal: whatsapp
Escribe XEND para salir.
--------------------------------------------------------------------------------
Agente: Hola, soy el asistente virtual de cobranzas. Para revisar tu caso, indícame tu DNI de 8 dígitos.
--------------------------------------------------------------------------------
Cliente: 12345678

Agente: Ya tengo tu DNI registrado como 12345678. ¿En qué puedo ayudarte hoy? Si deseas consultar tu deuda, revisar medios de pago o simular alternativas de pago, házmelo saber.
--------------------------------------------------------------------------------
Cliente: Cuanto debo?

Agente: Juan Pérez, tienes una deuda referencial de S/ 5,000.00 correspondiente a tu tarjeta de crédito. Actualmente, registras 45 días de atraso (mora media).

Si deseas, puedo ayudarte a revisar medios de pago o simular alternativas de pago. ¿Te gustaría proceder con alguna de estas opciones?
-----------------------